In [ ]:
# Colab / 新環境第一次執行時，取消下一行註解安裝套件。
# %pip install -q openai python-dotenv numpy chromadb pypdf python-docx pandas

import hashlib
import os

import numpy as np

print("Week 10 Embedding 環境載入完成 ✅")

In [ ]:
def local_demo_embed(text: str, dim: int = 256) -> list[float]:
    """離線假 embedding：用「字元 + bigram + 空白斷詞」雜湊成固定長度向量。

    這樣中文即使沒有空格，只要有共同字詞就會重疊（向量相近）。
    但這只是字面重疊、沒有真正語意；真正語意要用真 Embeddings API。
    """
    vector = [0.0] * dim
    lowered = text.lower()
    chars = [c for c in lowered if not c.isspace()]
    grams = list(chars)
    grams += [chars[i] + chars[i + 1] for i in range(len(chars) - 1)]
    grams += lowered.split()
    if not grams:
        grams = [lowered]
    for gram in grams:
        bucket = int(hashlib.md5(gram.encode("utf-8")).hexdigest(), 16) % dim
        vector[bucket] += 1.0
    return vector


v = local_demo_embed("向量 語意 搜尋")
print(f"向量長度 = {len(v)}；非零元素數 = {sum(1 for x in v if x)}")

In [ ]:
def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """計算兩向量的 cosine 相似度；任一為零向量時回 0。

    TODO（核心技能 1）：
      1. 用 np.asarray(..., dtype=float) 把兩個向量轉成 numpy 陣列。
      2. 用 np.linalg.norm() 算兩者長度；任一為 0 就回傳 0.0（避免除以零）。
      3. 回傳 float(np.dot(a, b) / (norm_a * norm_b))。
    提示：完成前 run_local_checks() 會因斷言失敗而報錯，那是預期的自我檢查。
    """
    # TODO: 實作 cosine（先回傳 0.0）
    return 0.0


print("cosine_similarity 已定義")

In [ ]:
same = cosine_similarity(local_demo_embed("向量 語意 搜尋"),
                         local_demo_embed("向量 語意 搜尋"))
diff = cosine_similarity(local_demo_embed("向量 語意 搜尋"),
                         local_demo_embed("股票 投資 理財"))
print(f"相同文字的相似度 = {same:.3f}")
print(f"不同文字的相似度 = {diff:.3f}")

In [ ]:
def get_embedding(text: str, offline: bool = False, model: str | None = None) -> list[float]:
    """把單一段文字轉成向量。offline=True 時使用離線假 embedding。

    TODO（核心技能 2）：完成「真 Embeddings API」分支。
      提示：真 API 其實就是呼叫下方的 embed_texts([text])[0]。
    （offline 分支已提供，讓你沒有 API key 也能先跑後面的搜尋管線。）
    """
    if not text or not text.strip():
        raise ValueError("輸入文字為空，無法產生 embedding。")
    if offline:
        return local_demo_embed(text)   # 離線分支已提供
    # TODO: 回傳真 API 向量（用 embed_texts）
    raise NotImplementedError("請完成 get_embedding 的真 API 分支")


def embed_texts(texts: list[str], offline: bool = False, model: str | None = None) -> list[list[float]]:
    """批次把多段文字轉成向量（此函式已提供完整實作）。"""
    cleaned = [t for t in texts if t and t.strip()]
    if not cleaned:
        raise ValueError("沒有可產生 embedding 的非空文字。")
    if offline:
        return [local_demo_embed(t) for t in cleaned]

    from openai import OpenAI
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("找不到 OPENAI_API_KEY；請先設定環境變數或 Colab Secret。")
    client = OpenAI(api_key=api_key)
    selected_model = model or os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")
    try:
        response = client.embeddings.create(model=selected_model, input=cleaned)
    except Exception as exc:
        raise RuntimeError(f"呼叫 Embeddings API 失敗：{exc}") from exc
    return [item.embedding for item in response.data]


print("get_embedding / embed_texts 已定義")

In [ ]:
# 付費 API 預設關閉。確認已設定測試用 API key 後，再改成 True。
RUN_PAID_API = False

if RUN_PAID_API:
    real_vec = get_embedding("生成式 AI 應用開發")
    print(f"真 embedding 維度 = {len(real_vec)}；前 5 維 = {[round(x, 4) for x in real_vec[:5]]}")
else:
    print("已略過付費 API；下面改用 offline=True 的離線假 embedding 練習搜尋流程。")

In [ ]:
def search_chunks(query: str, chunks: list[dict], top_k: int = 5, offline: bool = False) -> list[dict]:
    """最小語意搜尋：query 轉向量，與每個 chunk 的 embedding 算 cosine，回傳 top-k。

    TODO（核心技能 3）：
      1. query 為空要 raise ValueError；chunks 為空回傳 []。
      2. 用 get_embedding(query, offline=offline) 取得查詢向量。
      3. 對每個 chunk：用 cosine_similarity 算分數，組出「原欄位（去掉 embedding）+ score」。
      4. 依 score 由大到小排序，回傳前 top_k 筆。
    提示：先回傳 []，讓下游 demo 不會壞；完成後再看排序結果。
    """
    # TODO: 實作語意搜尋
    return []


print("search_chunks 已定義")

In [ ]:
demo_docs = [
    "Embedding 是把文字轉成向量，語意相近向量也相近。",
    "向量資料庫把向量、原文與 metadata 存在一起，支援快速查詢。",
    "今天天氣晴朗，很適合到公園散步與運動。",
    "OpenAI Embeddings API 會依文字長度計費，建議批次處理。",
]
demo_chunks = [{"chunk_id": i, "text": t, "source": "demo"} for i, t in enumerate(demo_docs)]

# 用離線假 embedding 先算好每個 chunk 的向量（不花錢）
for chunk, vector in zip(demo_chunks, embed_texts([c["text"] for c in demo_chunks], offline=True)):
    chunk["embedding"] = vector

results = search_chunks("向量資料庫是什麼？", demo_chunks, top_k=3, offline=True)
print("查詢：向量資料庫是什麼？（離線假 embedding，僅示範流程）\n")
for hit in results:
    print(f"score={hit['score']}  chunk={hit['chunk_id']}  {hit['text'][:24]}")

In [ ]:
def run_local_checks() -> None:
    """不呼叫付費 API，檢查 cosine 與搜尋的關鍵性質。"""
    v_same_a = local_demo_embed("向量 語意 搜尋")
    v_same_b = local_demo_embed("向量 語意 搜尋")
    v_other = local_demo_embed("完全 不同 主題 內容")
    assert abs(cosine_similarity(v_same_a, v_same_b) - 1.0) < 1e-6, "相同文字相似度應為 1"
    assert cosine_similarity(v_same_a, v_other) < cosine_similarity(v_same_a, v_same_b), "不同應小於相同"
    assert cosine_similarity([0, 0, 0], [1, 2, 3]) == 0.0, "零向量相似度應為 0"

    docs = [{"chunk_id": i, "text": t} for i, t in enumerate(["蘋果 是 水果", "股票 與 投資", "蘋果 水果 很甜"])]
    for d in docs:
        d["embedding"] = local_demo_embed(d["text"])
    hits = search_chunks("蘋果 水果", docs, top_k=2, offline=True)
    assert len(hits) == 2, "top_k=2 應回兩筆"
    assert hits[0]["score"] >= hits[1]["score"], "結果應由高到低排序"

    print("✅ 本機檢查通過：未呼叫付費 API")


run_local_checks()

In [ ]:
def build_chroma_collection(chunks: list[dict], offline: bool = False, name: str = "week10_docs"):
    """練習 A：把 chunks 建成 ChromaDB collection（in-memory，cosine 空間）。

    TODO：
      1. chunks 為空要 raise ValueError；import chromadb。
      2. texts = 每個 chunk 的 text；用 embed_texts(texts, offline=offline) 產生向量。
      3. client = chromadb.EphemeralClient()（可先 try/except 刪掉同名 collection）。
      4. collection = client.create_collection(name, metadata={"hnsw:space": "cosine"})。
      5. collection.add(ids=..., documents=..., embeddings=..., metadatas=...)：
         ids 用 str(chunk_id)、metadata 至少含 chunk_id 與 source。
      6. 回傳 collection。
    提示：先回傳 None，讓下游 demo 不會壞；完成後再執行。
    """
    # TODO: 實作 ChromaDB 索引建立
    return None


demo_collection = build_chroma_collection(demo_chunks, offline=True)
print("collection：", demo_collection, "（完成 TODO 後會是一個 collection 物件）")

In [ ]:
def query_chroma(collection, query: str, top_k: int = 5, offline: bool = False) -> list[dict]:
    """練習 B：對 collection 做 top-k 查詢，回傳含來源與相似度的結果。

    TODO：
      1. query 為空要 raise ValueError。
      2. 用 get_embedding(query, offline=offline) 取得查詢向量。
      3. result = collection.query(query_embeddings=[list(query_vector)], n_results=top_k)。
      4. 從 result 取出 documents/metadatas/distances（各為 [[...]]，取 [0]）。
      5. 逐筆組成 dict：text、metadata、score=round(1 - distance, 4)，回傳清單。
    提示：先回傳 []，完成後才看得到查詢結果與來源。
    """
    # TODO: 實作 ChromaDB 查詢
    return []


if demo_collection is not None:
    for hit in query_chroma(demo_collection, "向量資料庫怎麼運作", top_k=3, offline=True):
        print(f"score={hit['score']}  source={hit['metadata'].get('source')}  {hit['text'][:24]}")
else:
    print("請先完成練習 A 的 build_chroma_collection。")

In [ ]:
# 練習 C：先規劃再動手。把你的改造計畫填進這個 dict。
challenge_plan = {
    "feature": "",       # TODO: 你要新增的功能名稱
    "input": "",         # TODO: 需要什麼輸入
    "output": "",        # TODO: 會產生什麼輸出
    "error_cases": [],   # TODO: 至少列兩個要處理的錯誤情況
    "manual_test": "",   # TODO: 你會怎麼手動測試
}
import json as _json
print(_json.dumps(challenge_plan, ensure_ascii=False, indent=2))